In [ ]:
# ADLS Connection Configuration
storage_account = 
client_id       = 
tenant_id       = 
client_secret   = 

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("ADLS Connected ")

In [ ]:
tables = [
    "orders",
    "order_details", 
    "customers",
    "products",
    "categories",
    "shippers",
    "employees"
]

bronze_path = "abfss://ady-adf-practice@adlsadylearning.dfs.core.windows.net/NorthWind Project/Bronze"

from pyspark.sql.functions import col, sum as spark_sum

for table in tables:
    print(f"\n{'='*50}")
    print(f"TABLE: {table}")
    print(f"{'='*50}")
    
    df = spark.read.option("recursiveFileLookup", "true") \
                   .parquet(f"{bronze_path}/{table}/")
    
    # Schema
    print("\n--- SCHEMA ---")
    df.printSchema()
    
    # Row count
    print(f"Row count: {df.count()}")
    
    # Null check
    print("\n--- NULL COUNTS ---")
    df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()
    
    # Sample
    print("\n--- SAMPLE (3 rows) ---")
    df.show(3, truncate=False)

In [ ]:
from pyspark.sql.functions import col, to_date, lit, when, current_timestamp

# ── CONFIG ──────────────────────────────────────────
storage_account = "adlsadylearning"
container       = "ady-adf-practice"
bronze_path     = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Bronze"
silver_path     = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Silver"

# ── READ BRONZE ─────────────────────────────────────
df_orders        = spark.read.option("recursiveFileLookup", "true").parquet(f"{bronze_path}/orders/")
print("Bronze row count:", df_orders.count())

# ── TRANSFORMATIONS ─────────────────────────────────

# 1. Cast date strings to proper date type
df_orders = df_orders.withColumn("orderDate",    to_date(col("orderDate"),    "yyyy-MM-dd")) \
                     .withColumn("requiredDate", to_date(col("requiredDate"), "yyyy-MM-dd")) \
                     .withColumn("shippedDate",  to_date(col("shippedDate"),  "yyyy-MM-dd"))

# 2. Cast IDs to integer
df_orders = df_orders.withColumn("orderID",    col("orderID").cast("integer")) \
                     .withColumn("employeeID", col("employeeID").cast("integer")) \
                     .withColumn("shipperID",  col("shipperID").cast("integer"))

# 3. Rename columns to snake_case
df_orders = df_orders.withColumnRenamed("orderID",      "order_id") \
                     .withColumnRenamed("customerID",   "customer_id") \
                     .withColumnRenamed("employeeID",   "employee_id") \
                     .withColumnRenamed("orderDate",    "order_date") \
                     .withColumnRenamed("requiredDate", "required_date") \
                     .withColumnRenamed("shippedDate",  "shipped_date") \
                     .withColumnRenamed("shipperID",    "shipper_id")

# 4. Deduplication
df_orders = df_orders.dropDuplicates()

# 5. Audit columns
df_orders = df_orders.withColumn("created_at",  current_timestamp()) \
                     .withColumn("updated_at",  current_timestamp()) \
                     .withColumn("source_file", lit("Bronze/orders.parquet")) \
                     .withColumn("pipeline",    lit("Northwind Incremental"))

# ── VALIDATE ────────────────────────────────────────
print("Silver row count:", df_orders.count())
df_orders.printSchema()
df_orders.show(5)

# ── WRITE SILVER ────────────────────────────────────
df_orders.write.format("delta") \
               .mode("overwrite") \
               .save(f"{silver_path}/orders")

print("silver.orders written successfully")

In [ ]:
from pyspark.sql.functions import *

# ── READ BRONZE ─────────────────────────────────────
df_order_details = spark.read.option("recursiveFileLookup", "true").parquet(f"{bronze_path}/order_details/")
print("Bronze row count:", df_order_details.count())

# ── TRANSFORMATIONS ─────────────────────────────────

# 1. Cast types
df_order_details = df_order_details.withColumn("orderID",   col("orderID").cast("integer")) \
                                   .withColumn("productID", col("productID").cast("integer")) \
                                   .withColumn("quantity",  col("quantity").cast("integer")) \
                                   .withColumn("unitPrice", col("unitPrice").cast("decimal(10,2)")) \
                                   .withColumn("discount",  col("discount").cast("decimal(4,2)"))

# 2. Add derived revenue column
df_order_details = df_order_details.withColumn("revenue",
                    round(col("unitPrice") * col("quantity") * (1 - col("discount")), 2))

# 3. Rename columns
df_order_details = df_order_details.withColumnRenamed("orderID",   "order_id") \
                                   .withColumnRenamed("productID", "product_id") \
                                   .withColumnRenamed("unitPrice", "unit_price")

# 4. Deduplication
df_order_details = df_order_details.dropDuplicates()

# 5. Audit columns
df_order_details = df_order_details.withColumn("created_at",  current_timestamp()) \
                                   .withColumn("updated_at",  current_timestamp()) \
                                   .withColumn("source_file", lit("Bronze/order_details.parquet")) \
                                   .withColumn("pipeline",    lit("northwind_silver"))

# ── VALIDATE ────────────────────────────────────────
print("Silver row count:", df_order_details.count())
df_order_details.printSchema()
df_order_details.show(5)

# ── WRITE SILVER ────────────────────────────────────
df_order_details.write.format("delta") \
                      .mode("overwrite") \
                      .save(f"{silver_path}/order_details")

print("silver.order_details written successfully")

In [ ]:
# ── READ BRONZE ─────────────────────────────────────
df_customers     = spark.read.option("recursiveFileLookup", "true").parquet(f"{bronze_path}/customers/")
print("Bronze row count:", df_customers.count())

# ── TRANSFORMATIONS ─────────────────────────────────

# 1. Trim whitespace from all string columns
for c in df_customers.columns:
    df_customers = df_customers.withColumn(c, trim(col(c)))

# 2. Standardize city and country to title case
df_customers = df_customers.withColumn("city",    initcap(col("city"))) \
                           .withColumn("country", initcap(col("country")))

# 3. Rename columns
df_customers = df_customers.withColumnRenamed("customerID",   "customer_id") \
                           .withColumnRenamed("companyName",  "company_name") \
                           .withColumnRenamed("contactName",  "contact_name") \
                           .withColumnRenamed("contactTitle", "contact_title")

# 4. Deduplication
df_customers = df_customers.dropDuplicates()

# 5. Audit columns
df_customers = df_customers.withColumn("created_at",  current_timestamp()) \
                           .withColumn("updated_at",  current_timestamp()) \
                           .withColumn("source_file", lit("Bronze/customers.parquet")) \
                           .withColumn("pipeline",    lit("northwind_silver"))

# ── VALIDATE ────────────────────────────────────────
print("Silver row count:", df_customers.count())
df_customers.printSchema()
df_customers.show(5)

# ── WRITE SILVER ────────────────────────────────────
df_customers.write.format("delta") \
                  .mode("overwrite") \
                  .save(f"{silver_path}/customers")

print("silver.customers written successfully")

In [ ]:
# ── READ BRONZE ─────────────────────────────────────
df_products      = spark.read.option("recursiveFileLookup", "true").parquet(f"{bronze_path}/products/")
print("Bronze row count:", df_products.count())

# ── TRANSFORMATIONS ─────────────────────────────────

# 1. Cast types
df_products = df_products.withColumn("productID",  col("productID").cast("integer")) \
                         .withColumn("categoryID", col("categoryID").cast("integer")) \
                         .withColumn("unitPrice",  col("unitPrice").cast("decimal(10,2)"))

# 2. Cast discontinued long → boolean
df_products = df_products.withColumn("discontinued",
                when(col("discontinued") == 1, True).otherwise(False))

# 3. Trim string columns
df_products = df_products.withColumn("productName",     trim(col("productName"))) \
                         .withColumn("quantityPerUnit", trim(col("quantityPerUnit")))

# 4. Rename columns
df_products = df_products.withColumnRenamed("productID",       "product_id") \
                         .withColumnRenamed("productName",     "product_name") \
                         .withColumnRenamed("quantityPerUnit", "quantity_per_unit") \
                         .withColumnRenamed("unitPrice",       "unit_price") \
                         .withColumnRenamed("categoryID",      "category_id")

# 5. Deduplication
df_products = df_products.dropDuplicates()

# 6. Audit columns
df_products = df_products.withColumn("created_at",  current_timestamp()) \
                         .withColumn("updated_at",  current_timestamp()) \
                         .withColumn("source_file", lit("Bronze/products.parquet")) \
                         .withColumn("pipeline",    lit("northwind_silver"))

# ── VALIDATE ────────────────────────────────────────
print("Silver row count:", df_products.count())
df_products.printSchema()
df_products.show(5)

# ── WRITE SILVER ────────────────────────────────────
df_products.write.format("delta") \
                 .mode("overwrite") \
                 .save(f"{silver_path}/products")

print("silver.products written successfully")

In [ ]:
# ── READ BRONZE ─────────────────────────────────────
df_categories    = spark.read.option("recursiveFileLookup", "true").parquet(f"{bronze_path}/categories/")
print("Bronze row count:", df_categories.count())

# ── TRANSFORMATIONS ─────────────────────────────────

# 1. Cast types
df_categories = df_categories.withColumn("categoryID", col("categoryID").cast("integer"))

# 2. Trim string columns
df_categories = df_categories.withColumn("categoryName", trim(col("categoryName"))) \
                             .withColumn("description",  trim(col("description")))

# 3. Rename columns
df_categories = df_categories.withColumnRenamed("categoryID",   "category_id") \
                             .withColumnRenamed("categoryName", "category_name")

# 4. Deduplication
df_categories = df_categories.dropDuplicates()

# 5. Audit columns
df_categories = df_categories.withColumn("created_at",  current_timestamp()) \
                             .withColumn("updated_at",  current_timestamp()) \
                             .withColumn("source_file", lit("Bronze/categories.parquet")) \
                             .withColumn("pipeline",    lit("northwind_silver"))

# ── VALIDATE ────────────────────────────────────────
print("Silver row count:", df_categories.count())
df_categories.printSchema()
df_categories.show(5)

# ── WRITE SILVER ────────────────────────────────────
df_categories.write.format("delta") \
                   .mode("overwrite") \
                   .save(f"{silver_path}/categories")

print("silver.categories written successfully")

In [ ]:
# ── READ BRONZE ─────────────────────────────────────
df_shippers      = spark.read.option("recursiveFileLookup", "true").parquet(f"{bronze_path}/shippers/")
print("Bronze row count:", df_shippers.count())

# ── TRANSFORMATIONS ─────────────────────────────────

# 1. Cast types
df_shippers = df_shippers.withColumn("shipperID", col("shipperID").cast("integer"))

# 2. Trim string columns
df_shippers = df_shippers.withColumn("companyName", trim(col("companyName")))

# 3. Rename columns
df_shippers = df_shippers.withColumnRenamed("shipperID",   "shipper_id") \
                         .withColumnRenamed("companyName", "company_name")

# 4. Deduplication
df_shippers = df_shippers.dropDuplicates()

# 5. Audit columns
df_shippers = df_shippers.withColumn("created_at",  current_timestamp()) \
                         .withColumn("updated_at",  current_timestamp()) \
                         .withColumn("source_file", lit("Bronze/shippers.parquet")) \
                         .withColumn("pipeline",    lit("northwind_silver"))

# ── VALIDATE ────────────────────────────────────────
print("Silver row count:", df_shippers.count())
df_shippers.printSchema()
df_shippers.show(5)

# ── WRITE SILVER ────────────────────────────────────
df_shippers.write.format("delta") \
                 .mode("overwrite") \
                 .save(f"{silver_path}/shippers")

print("silver.shippers written successfully")

In [ ]:
# ── READ BRONZE ─────────────────────────────────────
df_employees     = spark.read.option("recursiveFileLookup", "true").parquet(f"{bronze_path}/employees/")
print("Bronze row count:", df_employees.count())

# ── TRANSFORMATIONS ─────────────────────────────────

# 1. Cast types
df_employees = df_employees.withColumn("employeeID", col("employeeID").cast("integer")) \
                           .withColumn("reportsTo",  col("reportsTo").cast("integer"))

# 2. Fill null reportsTo — top of hierarchy has no manager
df_employees = df_employees.fillna({"reportsTo": 0})

# 3. Split employeeName into first and last name
df_employees = df_employees.withColumn("first_name", split(col("employeeName"), " ")[0]) \
                           .withColumn("last_name",  split(col("employeeName"), " ")[1])

# 4. Trim and standardize strings
df_employees = df_employees.withColumn("title",   trim(col("title"))) \
                           .withColumn("city",    initcap(trim(col("city")))) \
                           .withColumn("country", initcap(trim(col("country"))))

# 5. Rename columns
df_employees = df_employees.withColumnRenamed("employeeID",   "employee_id") \
                           .withColumnRenamed("employeeName", "employee_name") \
                           .withColumnRenamed("reportsTo",    "reports_to")

# 6. Reorder columns
df_employees = df_employees.select("employee_id", "employee_name", "first_name", 
                                   "last_name", "title", "city", "country", "reports_to")

# 7. Deduplication
df_employees = df_employees.dropDuplicates()

# 8. Audit columns
df_employees = df_employees.withColumn("created_at",  current_timestamp()) \
                           .withColumn("updated_at",  current_timestamp()) \
                           .withColumn("source_file", lit("Bronze/employees.parquet")) \
                           .withColumn("pipeline",    lit("northwind_silver"))

# ── VALIDATE ────────────────────────────────────────
print("Silver row count:", df_employees.count())
df_employees.printSchema()
df_employees.show(5)

# ── WRITE SILVER ────────────────────────────────────
df_employees.write.format("delta") \
                  .mode("overwrite") \
                  .save(f"{silver_path}/employees")

print("silver.employees written successfully")